# Activation function inductive biases

This notebook reproduces the context-dependent decision-making (CDDM) experiments from

> Tolmachev & Engel (2025). *Single-unit activations confer inductive biases for emergent circuit solutions to cognitive tasks.* Nature Machine Intelligence.

We train six architectures — `{Dale, no Dale} × {relu, sigmoid, tanh}` — and compare their population trajectories and single-unit selectivity with MDS embeddings.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import MDS

# Add NeuralRNN src to path
sys.path.insert(0, str(Path(".").resolve().parents[1] / "src"))

from neuralrnn import AutoConfig, AutoModel, Trainer, TrainingArguments
from neuralrnn.analysis import fit_pca
from neuralrnn.data import CognitiveTaskDataset
from neuralrnn.data.tasks.mante_task import generate_trials
from neuralrnn.train.objectives.base import Objective

# Checkpoint and figure directories for this notebook
MODEL_DIR = Path("./models/14")
FIG_DIR = Path("./figs/14")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


## 1. CDDM dataset


In [ ]:
def make_cddm_dataset(
    n_t=75,
    cohs=None,
    n_trials_per_condition=1,
    baseline=0.2,
    sigma_in=0.01,
    alpha=0.2,
    seed=0,
):
    """Create a CDDM dataset compatible with the training loop below.

    This is a thin wrapper around `generate_trials` from `mante_task` so the
    training function can request a fresh dataset per seed.
    """
    np.random.seed(seed)

    if cohs is None:
        cohs = [0.03, 0.06, 0.13, 0.25, 0.5, 1.0]
        cohs = sorted(set([-c for c in cohs if c != 0] + cohs))

    inputs, targets, mask, conditions = generate_trials(
        n_reps=n_trials_per_condition,
        alpha=alpha,
        sigma_in=sigma_in,
        baseline=baseline,
        cohs=cohs,
        n_t=n_t,
    )

    ds = CognitiveTaskDataset(
        inputs=inputs, targets=targets, mask=mask,
        conditions=conditions, task_name="mante_cddm",
        batch_size=inputs.shape[0],
    )
    return ds


cohs = [0.03, 0.06, 0.13, 0.25, 0.5, 1.0]
cohs = sorted(set([-c for c in cohs if c != 0] + cohs))

# Training set: one trial per condition, full batch for stable Adam updates.
ds = make_cddm_dataset(n_trials_per_condition=1, alpha=0.2, sigma_in=0.01, baseline=0.2, cohs=cohs, seed=0)

# Evaluation set: three trials per condition for richer trajectory analysis.
ds_eval = make_cddm_dataset(n_trials_per_condition=3, alpha=0.2, sigma_in=0.01, baseline=0.2, cohs=cohs, seed=1)

print(f"train inputs:  {tuple(ds.inputs.shape)}")
print(f"train targets: {tuple(ds.targets.shape)}")
print(f"train mask:    {tuple(ds.mask.shape)}")
print(f"eval inputs:   {tuple(ds_eval.inputs.shape)}")
print(f"eval targets:  {tuple(ds_eval.targets.shape)}")
print(f"eval mask:     {tuple(ds_eval.mask.shape)}")


### Dataset structure and visualization

The CDDM task used here is the `mante` generator from `neuralrnn.data.tasks`. Each trial has:

- **Input channels** (`6` total):
  - `motion_ctx` / `color_ctx`: context cue that tells the network which modality to attend to.
  - `motion_r` / `motion_l`: right / left motion evidence.
  - `color_r` / `color_l`: right / left color evidence.
- **Target channels** (`2` total): `choice_r` / `choice_l`, active only during the decision window.
- **Mask**: `1` during the decision period, `0` otherwise, so the supervised loss is evaluated only on choice-producing timesteps.

The panel below shows one example trial from each context (motion and color) so the temporal structure can be inspected.

In [ ]:
def plot_example_trial(ds, trial_idx, ax_in, ax_out):
    """Plot a single CDDM trial's inputs and target outputs."""
    inputs = ds.inputs[trial_idx].cpu().numpy()
    targets = ds.targets[trial_idx].cpu().numpy()
    mask = ds.mask[trial_idx, :, 0].cpu().numpy()
    cond = ds.conditions[trial_idx]

    ax_in.plot(inputs[:, 0], label="motion_ctx", color="C0")
    ax_in.plot(inputs[:, 1], label="color_ctx", color="C1")
    ax_in.plot(inputs[:, 2] - inputs[:, 3], label="motion evidence (r - l)", color="C2")
    ax_in.plot(inputs[:, 4] - inputs[:, 5], label="color evidence (r - l)", color="C3")
    ax_in.set_title(f"{cond['context']} context | motion_coh={cond['motion_coh']:.2f}, color_coh={cond['color_coh']:.2f}")
    ax_in.set_xlabel("Time step")
    ax_in.set_ylabel("Input")
    ax_in.legend(fontsize=7, loc="upper left")

    ax_out.plot(targets[:, 0], label="choice_r", color="C4")
    ax_out.plot(targets[:, 1], label="choice_l", color="C5")
    ax_out.fill_between(np.arange(len(mask)), 0, mask * targets.max(), color="gray", alpha=0.2, label="loss mask")
    ax_out.set_xlabel("Time step")
    ax_out.set_ylabel("Target output")
    ax_out.legend(fontsize=7, loc="upper left")


# Pick one motion-context trial and one color-context trial with strong, conflicting evidence.
motion_idx = next(i for i, c in enumerate(ds.conditions) if c["context"] == "motion" and c["motion_coh"] == 1.0 and c["color_coh"] == -1.0)
color_idx = next(i for i, c in enumerate(ds.conditions) if c["context"] == "color" and c["color_coh"] == 1.0 and c["motion_coh"] == -1.0)

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
plot_example_trial(ds, motion_idx, axes[0, 0], axes[0, 1])
plot_example_trial(ds, color_idx, axes[1, 0], axes[1, 1])
fig.suptitle("Example CDDM trials (strong relevant evidence, strong opposite irrelevant evidence)")
fig.tight_layout()
fig.savefig(FIG_DIR / "cddm_example_trials.png", dpi=150, bbox_inches='tight')
plt.show()


## 2. Train networks

* Architecture grid: `{relu, sigmoid, tanh} × {Dale, no Dale}`, 3 seeds each.

* Models are saved under `models/14/`. If a checkpoint already exists, we skip re-training and load it.

* We enforce non-negativity of `W_inp`/`W_out` with a `Trainer.post_step_hook`.

* Note that The paper's supervised loss for CDDM is

    $$\mathcal{L} = \underbrace{\mathrm{MSE}_{\mathrm{mask}}(W_{\mathrm{out}} y, \mathrm{target})}_{\text{masked output error}} \;+\; \lambda_r \underbrace{\langle \|y\|_2^2 \rangle}_{\text{mean firing-rate penalty}} \;+\; \lambda_\perp \underbrace{\|W_{\mathrm{inp}}^\top W_{\mathrm{inp}} - \mathrm{diag}(W_{\mathrm{inp}}^\top W_{\mathrm{inp}})\|_2^2}_{\text{input orthogonality penalty}}$$

    For generalization, we just use the MSE objective function (the first item), but the implementation of this OrthogonalityObjective can be found in notebook 02. 

In [ ]:
ARCHITECTURES = [
    ("relu", False, "ctrnn", None),
    ("relu", True, "ei_rnn", 0.8),
    ("sigmoid", False, "ctrnn", None),
    ("sigmoid", True, "ei_rnn", 0.8),
    ("tanh", False, "ctrnn", None),
    ("tanh", True, "ei_rnn", 0.5),
]
SEEDS = [0, 1, 2]
SAVE_ROOT = Path(".").resolve() / "models" / "14"


def make_nonneg_clamp_hook():
    def hook(model):
        with torch.no_grad():
            model.input2h.weight.clamp_(min=0.0)
            model.readout_layer.weight.clamp_(min=0.0)
    return hook

def train_one(activation, dale, model_type, ei_ratio, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    kwargs = dict(
        input_dim=6, latent_dim=100, output_dim=2,
        dt=2.0, tau=10.0, activation=activation,
        sigma_rec=0.05, nonlinearity_mode='pre_activation',
    )
    if dale:
        kwargs.update(dale=True, ei_ratio=ei_ratio, readout_e_only=False)
        cfg = AutoConfig.for_model("ei_rnn", **kwargs)
    else:
        cfg = AutoConfig.for_model("ctrnn", **kwargs)

    model = AutoModel.from_config(cfg)
    # init_activation_matters_weights(model, seed=seed)

    ds = make_cddm_dataset(seed=seed)
    
    # objective = OrthogonalityObjective()
    from neuralrnn.train.objectives import SupervisedObjective
    objective = SupervisedObjective('regression')
    args = TrainingArguments(
        max_steps=4000, learning_rate=0.005, weight_decay=5e-6,
        optimizer="adam", grad_clip_norm=None, log_every=200,
        device=DEVICE, seed=seed,
    )
    trainer = Trainer(model, ds, objective, args, post_step_hook=make_nonneg_clamp_hook())
    trainer.train()
    return model


def get_or_train(activation, dale, model_type, ei_ratio, seed):
    save_dir = SAVE_ROOT / f"{model_type}_{activation}_dale={dale}_seed{seed}"
    if save_dir.exists():
        print(f"Loading {save_dir.name}")
        return AutoModel.from_pretrained(str(save_dir)), True
    print(f"Training {save_dir.name}")
    model = train_one(activation, dale, model_type, ei_ratio, seed)
    model.save_pretrained(str(save_dir))
    return model, False


models = []
for activation, dale, model_type, ei_ratio in ARCHITECTURES:
    for seed in SEEDS:
        model, loaded = get_or_train(activation, dale, model_type, ei_ratio, seed)
        models.append({
            "model": model,
            "activation": activation,
            "dale": dale,
            "model_type": model_type,
            "seed": seed,
            "loaded": loaded,
        })

print(f"Total models: {len(models)}")


### Test accuracy

Compute the fraction of trials on which each trained network makes the correct choice. 

In [ ]:
@torch.no_grad()
def evaluate_accuracy(model, ds, device=DEVICE):
    """Evaluate CDDM decision accuracy on a dataset.

    For each trial we average the two readout channels over the masked decision
    window and take the channel with the larger average as the network's choice.
    The correct choice is inferred from the condition dictionary:
    positive coherence in the relevant modality -> right choice (channel 0),
    negative coherence -> left choice (channel 1).

    Returns
    -------
    acc : float
        Overall fraction of correct choices.
    acc_by_context : dict
        Accuracy split by attended context ('motion' / 'color').
    mse : float
        Masked mean-squared error on the test set.
    """
    model.to(device)
    model.eval()
    out = model(ds.inputs.to(device))
    y = out.outputs  # (B, T, 2)

    # The decision mask is the same for both output channels in this task.
    dec_mask = ds.mask[:, :, 0].bool().to(device)
    # Average each output channel over the decision window per trial.
    dec_out = (y * dec_mask.unsqueeze(-1)).sum(dim=1) / dec_mask.sum(dim=1, keepdim=True).clamp_min(1.0)
    pred_choice = dec_out.argmax(dim=1).cpu()

    # Map condition to true choice: 0 = right, 1 = left.
    true_choice = torch.tensor([
        0 if c["correct_choice"] == 1 else 1 for c in ds.conditions
    ])

    overall_acc = (pred_choice == true_choice).float().mean().item()

    contexts = np.array([c["context"] for c in ds.conditions])
    acc_by_context = {}
    for ctx in ["motion", "color"]:
        idx = np.where(contexts == ctx)[0]
        acc_by_context[ctx] = (pred_choice[idx] == true_choice[idx]).float().mean().item()

    err = (y.cpu() - ds.targets) ** 2
    mse = (err * ds.mask).sum() / ds.mask.sum()

    return overall_acc, acc_by_context, mse.item()


print("Test-set performance per architecture (mean ± std over 3 seeds):\n")
print(f"{'Arch':<25s} {'Overall Acc':>12s} {'Motion Acc':>12s} {'Color Acc':>12s} {'MSE':>10s}")
print("-" * 75)
for activation, dale, model_type, ei_ratio in ARCHITECTURES:
    overall, motion, color, mse = [], [], [], []
    for seed in SEEDS:
        info = next(m for m in models
                    if m["activation"] == activation and m["dale"] == dale and m["seed"] == seed)
        oa, abc, mse_i = evaluate_accuracy(info["model"], ds_eval)
        overall.append(oa)
        motion.append(abc["motion"])
        color.append(abc["color"])
        mse.append(mse_i)
    label = f"{activation} Dale={dale}"
    print(f"{label:<25s} {np.mean(overall):>12.3f} {np.mean(motion):>12.3f} {np.mean(color):>12.3f} {np.mean(mse):>10.4f}")
print("\nNote: Low accuracy with sigmoid activation is because the original work use a slope of 7.5")
print("It can be achieved by defining new activations")



## 3. Model Analysis

This section implements the feature extraction and distance metrics used for the MDS embeddings in Tolmachev & Engel (2025). Each helper is documented with its mathematical meaning, input shapes, and the role of every variable.

### Notation

For a single trained network we collect the post-activation (post-blend) state trajectory

$$y \in \mathbb{R}^{N \times T \times K}$$

where:
- $N$ = number of recurrent units (`latent_dim`, here $N=100$),
- $T$ = number of time steps per trial,
- $K$ = number of trials in the analysis batch.

The tensor is indexed as `y[unit, time, trial]`. All PCA projections use `neuralrnn.analysis.fit_pca`, which wraps scikit-learn's PCA and stores the trained basis.

### (i) Population-trajectory feature

The population-trajectory representation is obtained by **flattening time and trials** and projecting the unit activity onto the first $n_\text{PC}=10$ principal components:

1. Reshape $y$ to $\mathbf{Y} \in \mathbb{R}^{(T \cdot K) \times N}$ (each row is the $N$-dimensional population state at one `(time, trial)`).
2. Fit PCA on these rows and keep $n_\text{PC}=10$ components, obtaining $\mathbf{P}_\text{traj} \in \mathbb{R}^{N \times n_\text{PC}}$.
3. Project: $\mathbf{F}_\text{traj} = \mathbf{P}_\text{traj}^\top \mathbf{Y}^\top \in \mathbb{R}^{n_\text{PC} \times T \times K}$.
4. Normalize by the total variance: $R = \sqrt{\sum_{d=1}^{n_\text{PC}} \mathrm{Var}[\mathbf{F}_\text{traj}(d, \cdot)]}$ and divide $\mathbf{F}_\text{traj}$ by $R$.

The distance between two trajectory features $\mathbf{F}_i$ and $\mathbf{F}_j$ is a **symmetric least-squares fit**:

$$d_\text{traj}(\mathbf{F}_i, \mathbf{F}_j) = \frac{1}{2}\bigl(\|\mathbf{A}_i \mathbf{M}_i - \mathbf{A}_j\|_F + \|\mathbf{A}_j \mathbf{M}_j - \mathbf{A}_i\|_F\bigr)$$

where $\mathbf{A}_i = \mathbf{F}_i^{(n_\text{PC} \times T K)}$ is the feature reshaped to $(n_\text{PC}, T\cdot K)$ and $\mathbf{M}_i = \arg\min_\mathbf{M} \|\mathbf{A}_i \mathbf{M} - \mathbf{A}_j\|_F$ is the best linear map from the PC space of network $i$ to that of network $j$.

### (ii) Single-unit selectivity feature

Single-unit selectivity captures how each unit responds across the full set of `(time, trial)` conditions. We keep the units as the rows and reduce the $T\cdot K$ condition dimension:

1. Reshape $y$ to $\mathbf{S} \in \mathbb{R}^{N \times (T \cdot K)}$.
2. Fit PCA on these rows and keep $n_\text{PC}=10$ components, giving selectivities $\mathbf{S}_\text{proj} \in \mathbb{R}^{N \times n_\text{PC}}$.
3. Center and scale: subtract the mean across units and divide by $\sqrt{\sum_d \mathrm{Var}[\mathbf{S}_\text{proj}(:, d)]}$.

Because the PC axes of two networks are not aligned, we use **Iterative Closest Point (ICP) registration** under orthogonal transformations to measure distance:

$$d_\text{sel}(\mathbf{S}_i, \mathbf{S}_j) = \frac{1}{2}\bigl(d_\text{ICP}(\mathbf{S}_i, \mathbf{S}_j) + d_\text{ICP}(\mathbf{S}_j, \mathbf{S}_i)\bigr)$$

ICP alternates between nearest-neighbor matching of rows (units) and orthogonal Procrustes alignment, restarted from 60 random rotations to avoid local minima.

### (iii) Trajectory-endpoint feature

Endpoints are simply the population state at the last time step of each trial:

$$\mathbf{E} = y[:, T-1, :] \in \mathbb{R}^{N \times K}$$

We project to $n_\text{PC}=10$ components, center, and scale as above. The endpoint distance is again a symmetric orthogonal Procrustes distance.

### (iv) Pairwise distance matrix and MDS

For each feature type we build an $18 \times 18$ symmetric distance matrix (6 architectures × 3 seeds). The matrix is passed to `sklearn.manifold.MDS` with `dissimilarity="precomputed"` to obtain a 2-D embedding used for visualization.

In [ ]:
# Data collection
from scipy.linalg import orthogonal_procrustes
from scipy.stats import ortho_group

@torch.no_grad()
def collect_trajectory(model, ds, device=DEVICE):
    """Collect recurrent-state trajectories from a trained network.
    Returns
    -------
    traj : np.ndarray, shape (N, T, K)
        Post-activation recurrent states, indexed as [unit, time, trial].
    conditions : list
        Trial condition dictionaries from the dataset.
    """
    model.to(device)
    model.eval()
    out = model(ds.inputs.to(device))
    # NeuralRNN returns states as (B, T, N); transpose to analytical convention (N, T, B).
    traj = out.states.cpu().numpy().transpose(2, 1, 0)
    return traj, ds.conditions


# 1 Population-trajectory feature and distance

def extract_trajectory_feature(traj, n_PCs=10):
    """Build the population-trajectory feature.
    Parameters
    ----------
    traj : np.ndarray, shape (N, T, K)
        Recurrent-state trajectories for one network.
    n_PCs : int
        Number of principal components to retain.
    Returns
    -------
    feat : np.ndarray, shape (n_PCs, T, K)
        Variance-normalized PC projection of the population trajectory.
    pca : fitted PCA object
        PCA fitted on the flattened (T*K, N) data matrix.
    """
    N, T, K = traj.shape
    # Y has shape (T*K, N): each row is the N-dimensional population state at one (time, trial).
    Y = traj.reshape(N, -1).T
    pca = fit_pca(Y, n_components=min(n_PCs, N))
    # Project and reshape back to (n_PCs, T, K).
    feat = pca.transform(Y).T.reshape(n_PCs, T, K)
    # R is the square root of the summed variance across the retained PCs.
    R = np.sqrt(np.sum(np.var(feat.reshape(n_PCs, -1), axis=1)))
    return feat / (R + 1e-12), pca


def trajectory_distance(Fi, Fj):
    """Symmetric least-squares distance between two trajectory features.

    Parameters
    ----------
    Fi, Fj : np.ndarray, shape (n_PCs, T, K)
        Variance-normalized trajectory features for networks i and j.

    Returns
    -------
    d : float
        d = 0.5 * (||Ai Mi - Aj||_F + ||Aj Mj - Ai||_F),
        where Ai, Aj are (n_PCs, T*K) and Mi, Mj are least-squares mappings.
    """
    # Reshape to (n_PCs, T*K) so columns are population-PC coordinates over all conditions.
    Ai = Fi.reshape(Fi.shape[0], -1).T
    Aj = Fj.reshape(Fj.shape[0], -1).T
    # Best linear map from i's PC space to j's PC space.
    M1, *_ = np.linalg.lstsq(Ai, Aj, rcond=None)
    s1 = np.sqrt(np.sum((Ai @ M1 - Aj) ** 2))
    # Symmetric term: best linear map from j to i.
    M2, *_ = np.linalg.lstsq(Aj, Ai, rcond=None)
    s2 = np.sqrt(np.sum((Aj @ M2 - Ai) ** 2))
    return (s1 + s2) / 2.0


# 2 Single-unit selectivity feature and ICP distance

def extract_selectivity_feature(traj, n_PCs=10):
    """Build the single-unit selectivity feature used for MDS.

    Parameters
    ----------
    traj : np.ndarray, shape (N, T, K)
        Recurrent-state trajectories for one network.
    n_PCs : int
        Number of principal components to retain along the condition axis
        (paper uses 10).

    Returns
    -------
    sel : np.ndarray, shape (N, n_PCs)
        Centered and variance-normalized selectivity matrix; each row is one
        unit's response profile projected onto the top PCs.
    """
    N, T, K = traj.shape
    # S has shape (N, T*K): each row is one unit's response across all (time, trial) conditions.
    S = traj.reshape(N, -1)
    # PCA is fit on rows, reducing the condition dimension from T*K to n_PCs.
    pca = fit_pca(S, n_components=min(n_PCs, N))
    sel = pca.transform(S)  # (N, n_PCs)
    # Center across units and normalize by the summed PC variance.
    sel = sel - sel.mean(axis=0)
    norm = np.sqrt(np.sum(np.var(sel, axis=0)))
    return sel / (norm + 1e-12)


def _pairwise_distance(A, B):
    """Euclidean distance matrix between two point clouds.

    Parameters
    ----------
    A : np.ndarray, shape (n, d)
    B : np.ndarray, shape (m, d)

    Returns
    -------
    D : np.ndarray, shape (n, m)
        D[i, j] = ||A[i] - B[j]||_2.
    """
    return np.sqrt(np.maximum(
        (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T, 0
    ))


def register_icp(Ps, Pt, n_tries=60, max_iter=1000, tol=1e-10, seed=0):
    """Register source point cloud Ps to target Pt using ICP under orthogonal maps.

    ICP iterates two steps:
      1. Nearest-neighbor matching: for each point in Ps @ Q find closest point in Pt.
      2. Orthogonal Procrustes: update Q to best align Ps with the matched targets.

    The procedure is restarted from `n_tries` random initial rotations and the
    lowest MSE alignment is returned.

    Parameters
    ----------
    Ps : np.ndarray, shape (n_source, d_source)
        Source point cloud (rows are units/selectivities).
    Pt : np.ndarray, shape (n_target, d_target)
        Target point cloud. d_source <= d_target is assumed.
    n_tries : int
        Number of random restarts (paper uses 60).
    max_iter : int
        Maximum ICP iterations per restart.
    tol : float
        Convergence tolerance on MSE change.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    best_Q : np.ndarray
        Best orthogonal alignment matrix.
    best_mse : float
        Lowest symmetric registration error achieved.
    """
    best_mse = np.inf
    best_Q = None
    for t in range(n_tries):
        # Random initial orthogonal matrix: sample from O(d_target) and truncate.
        Q = ortho_group.rvs(Pt.shape[1], random_state=seed + t)[:Ps.shape[1], :Pt.shape[1]]
        mse_prev = np.inf
        for _ in range(max_iter):
            # Match each transformed source point to its nearest target point.
            D = _pairwise_distance(Ps @ Q, Pt)
            Pt_matched = Pt[np.argmin(D, axis=1)]
            # Optimal orthogonal map for the current matching.
            Q, _ = orthogonal_procrustes(Ps, Pt_matched)
            mse = np.sqrt(np.sum((Ps @ Q - Pt_matched) ** 2))
            if mse_prev - mse < tol:
                break
            mse_prev = mse
        if mse < best_mse:
            best_mse = mse
            best_Q = Q
    return best_Q, best_mse


def icp_distance(Si, Sj, seed=0):
    """Symmetric ICP distance between two selectivity matrices.

    The distance is averaged over both registration directions to enforce
    symmetry: d(Si, Sj) = 0.5 * (ICP(Si -> Sj) + ICP(Sj -> Si)).
    """
    _, s1 = register_icp(Si, Sj, seed=seed)
    _, s2 = register_icp(Sj, Si, seed=seed + 1)
    return (s1 + s2) / 2.0


# 3 Trajectory-endpoint feature and distance
def extract_endpoint_feature(traj, n_PCs=10):
    """Build the trajectory-endpoint feature used for MDS.

    Parameters
    ----------
    traj : np.ndarray, shape (N, T, K)
        Recurrent-state trajectories for one network.
    n_PCs : int
        Number of principal components to retain (paper uses 10).

    Returns
    -------
    X : np.ndarray, shape (K, n_PCs)
        Variance-normalized projection of the last-trial states onto the top PCs.
    """
    N, T, K = traj.shape
    # E has shape (N, K): state at the final time step for each trial.
    E = traj[:, -1, :]
    pca = fit_pca(E.T, n_components=min(n_PCs, N))
    X = pca.transform(E.T)  # (K, n_PCs)
    X = X - X.mean(axis=0)
    R = np.sqrt(np.mean(np.sum(X ** 2, axis=1)))
    return X / (R + 1e-12)


def procrustes_distance(Pi, Pj):
    """Symmetric orthogonal Procrustes distance between two endpoint features.

    Parameters
    ----------
    Pi, Pj : np.ndarray, shape (K, n_PCs)
        Endpoint PC features for networks i and j.

    Returns
    -------
    d : float
        d = 0.5 * (||Pi Qi - Pj||_F + ||Pj Qj - Pi||_F) with orthogonal Qi, Qj.
    """
    Q1, _ = orthogonal_procrustes(Pi, Pj)
    s1 = np.sqrt(np.sum((Pi @ Q1 - Pj) ** 2))
    Q2, _ = orthogonal_procrustes(Pj, Pi)
    s2 = np.sqrt(np.sum((Pj @ Q2 - Pi) ** 2))
    return (s1 + s2) / 2.0


# 4 Pairwise distance matrix
def build_distance_matrix(features, metric_fn):
    """Build a symmetric pairwise distance matrix from a list of feature objects.

    Parameters
    ----------
    features : list
        One feature object per network (18 networks total).
    metric_fn : callable
        Distance function taking two feature objects and returning a scalar.

    Returns
    -------
    Mat : np.ndarray, shape (len(features), len(features))
        Symmetric distance matrix with zero diagonal.
    """
    n = len(features)
    Mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = metric_fn(features[i], features[j])
            Mat[i, j] = Mat[j, i] = d
    return Mat


## Extract features from all trained networks

We compute the three feature sets used in the MDS embeddings:
population trajectories, single-unit selectivities, and trajectory endpoints.
Features are stored in the same order as `models` so that the distance matrix
row/column indices match the model list.

In [ ]:
traj_features, sel_features, end_features = [], [], []
traj_list, conditions_list = [], []

for info in models:
    model = info["model"]
    traj, conds = collect_trajectory(model, ds_eval, device=DEVICE)
    traj_list.append(traj)
    conditions_list.append(conds)

    traj_feat, _ = extract_trajectory_feature(traj, n_PCs=10)
    traj_features.append(traj_feat)
    sel_features.append(extract_selectivity_feature(traj, n_PCs=10))
    end_features.append(extract_endpoint_feature(traj, n_PCs=10))

    print(f"{info['activation']:8s} Dale={info['dale']!s:5s} seed={info['seed']}: "
          f"traj={traj_features[-1].shape}, sel={sel_features[-1].shape}, end={end_features[-1].shape}")


## Visualize trajectories and single-unit selectivities

Before building distance matrices, we inspect the geometry of the learned representations. Following the original project, we create **one $2 \times 3$ panel per metric** (population trajectories, then single-unit selectivities). Each of the six subplots shows one architecture (`relu / sigmoid / tanh` × `Dale / no Dale`), using seed 0 as the representative network.

Trajectory colors follow the reference implementation:
- **Line color**: coherence of the *relevant* modality.
- **Marker / point color**: coherence of the *irrelevant* modality.
- **Marker shape**: `p` for motion context, `*` for color context.

This means the two contexts are separated by marker shape, and the five coherence levels are separated by color. For clarity the visualization uses a dedicated dataset with coherences `{-1, -0.5, 0, 0.5, 1}` (one trial per condition, 50 trials total).

In [ ]:
from scipy.interpolate import interp1d


def interpolate_color(low_color, mid_color, high_color, low_val, mid_val, high_val, val):
    """Linearly interpolate between three RGB colors using a scalar value.

    Replicates the color interpolation helper in the reference project.
    """
    val = np.clip(val, low_val, high_val)
    if val < mid_val:
        w0 = (mid_val - val) / (mid_val - low_val)
        w1 = (val - low_val) / (mid_val - low_val)
        return w0 * low_color + w1 * mid_color
    elif val > mid_val:
        w0 = (high_val - val) / (high_val - mid_val)
        w1 = (val - mid_val) / (high_val - mid_val)
        return w0 * mid_color + w1 * high_color
    else:
        return mid_color


def get_cddm_plotting_params(conditions):
    """Return per-trial colors and markers for CDDM trajectory plots.

    Returns
    -------
    line_colors : (K, 3) array
        Color of the trajectory line for each trial, encoding the coherence of
        the *relevant* modality.
    point_colors : (K, 3) array
        Color of the equidistant markers, encoding the coherence of the
        *irrelevant* modality.
    markers : list[str]
        Marker shape for each trial: 'p' = motion context, '*' = color context.
    """
    contexts = np.array([1 if c["context"] == "motion" else -1 for c in conditions])
    relevant_cohs = np.array([
        c["motion_coh"] if c["context"] == "motion" else c["color_coh"]
        for c in conditions
    ])
    irrelevant_cohs = np.array([
        c["color_coh"] if c["context"] == "motion" else c["motion_coh"]
        for c in conditions
    ])

    # Original diverging palette: blue -> gray -> red.
    primary_colors = np.array([[0.3, 0.4, 0.8], [0.8, 0.8, 0.8], [0.8, 0.2, 0.3]])

    def make_colors(values):
        low, high = values.min(), values.max()
        mid = (low + high) / 2.0
        return np.array([
            interpolate_color(primary_colors[0], primary_colors[1], primary_colors[2],
                              low, mid, high, v)
            for v in values
        ])

    line_colors = make_colors(relevant_cohs)
    point_colors = make_colors(irrelevant_cohs)
    markers = np.where(contexts == 1, "p", "*").tolist()
    return line_colors, point_colors, markers


def plot_single_trajectory(ax, traj_proj, line_color, point_color, marker,
                           axes=(0, 1), linewidth=0.75, markersize=30):
    """Plot one trial trajectory in the 2-D PC subspace.

    traj_proj has shape (n_PCs, T). The full temporal evolution is drawn as a
    line; 8 equidistant points along the path are overlaid as markers colored
    by the irrelevant-modality coherence.
    """
    traj = traj_proj[list(axes), :]  # (2, T)
    ax.plot(traj[0], traj[1], color=line_color, linewidth=linewidth, alpha=0.9)

    # Place 8 equidistant markers along the path.
    distances = np.sqrt(np.sum(np.diff(traj, axis=1) ** 2, axis=0))
    cumulative = np.insert(np.cumsum(distances), 0, 0)
    if cumulative[-1] == 0:
        return
    sample_s = np.linspace(0, cumulative[-1], 8)
    interp_x = interp1d(cumulative, traj[0], kind="linear")
    interp_y = interp1d(cumulative, traj[1], kind="linear")
    ax.scatter(interp_x(sample_s), interp_y(sample_s),
               color=point_color, facecolors=point_color,
               marker=marker, s=markersize, linewidth=0.1,
               edgecolors="k", alpha=0.9, zorder=3)


def plot_trajectory_panel(ax, traj_feature, conditions, axes=(0, 1)):
    """Plot all trials' trajectories into a single axis."""
    line_colors, point_colors, markers = get_cddm_plotting_params(conditions)
    K = traj_feature.shape[2]
    for k in range(K):
        plot_single_trajectory(ax, traj_feature[:, :, k],
                               line_color=line_colors[k],
                               point_color=point_colors[k],
                               marker=markers[k], axes=axes)
    ax.set_xticks([])
    ax.set_yticks([])


def plot_selectivity_panel(ax, sel_feature, axes=(0, 1)):
    """Plot the single-unit selectivity cloud into a single axis."""
    color = np.array([0.8, 0.2, 0.3])  # default color used in the reference project
    ax.scatter(sel_feature[:, axes[0]], sel_feature[:, axes[1]],
               edgecolors="k", color=color, s=20, alpha=0.5, linewidths=0.1)
    ax.set_xticks([])
    ax.set_yticks([])


# Build a visualization dataset
cohs_viz = [-1.0, -0.5, 0.0, 0.5, 1.0]
ds_viz = make_cddm_dataset(
    n_trials_per_condition=1,
    alpha=0.2,
    sigma_in=0.01,
    baseline=0.2,
    cohs=cohs_viz,
    seed=42,
)
print(f"viz dataset: {tuple(ds_viz.inputs.shape)} | {len(ds_viz.conditions)} trials")

# Verify the condition grid is complete and balanced.
conditions_set = set()
for c in ds_viz.conditions:
    conditions_set.add((c["context"], c["motion_coh"], c["color_coh"]))
print(f"unique (context, motion_coh, color_coh) combinations: {len(conditions_set)}")

# Pick one representative seed per architecture for visualization.
representative = {}
for info in models:
    key = (info["activation"], info["dale"])
    if key not in representative or info["seed"] == 0:
        representative[key] = info

# Order the six architectures for the 2x3 layout.
ARCH_ORDER = [
    ("relu", False), ("sigmoid", False), ("tanh", False),
    ("relu", True),  ("sigmoid", True),  ("tanh", True),
]

# Collect trajectories and selectivities on the visualization dataset.
viz_data = {}
for key in ARCH_ORDER:
    info = representative[key]
    traj, conds = collect_trajectory(info["model"], ds_viz, device=DEVICE)
    traj_feat, _ = extract_trajectory_feature(traj, n_PCs=10)
    sel_feat = extract_selectivity_feature(traj, n_PCs=10)
    viz_data[key] = {"traj": traj_feat, "sel": sel_feat, "conds": conds}


# Population-trajectory panel: 2 x 3, one subplot per architecture.
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
fig.suptitle("Population trajectories (PC1 vs PC2)", fontsize=14)

for ax, key in zip(axes.flat, ARCH_ORDER):
    activation, dale = key
    plot_trajectory_panel(ax, viz_data[key]["traj"], viz_data[key]["conds"])
    ax.set_title(f"{activation} — Dale={dale}")

axes[0, 0].set_ylabel("Motion / color context\nmarker shape")
axes[1, 0].set_ylabel("Motion / color context\nmarker shape")
fig.tight_layout()
fig.savefig(FIG_DIR / "population_trajectories_pcs.png", dpi=150, bbox_inches='tight')
plt.show()


# Single-unit selectivity panel: 2 x 3, one subplot per architecture.
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
fig.suptitle("Single-unit selectivities (PC1 vs PC2)", fontsize=14)

for ax, key in zip(axes.flat, ARCH_ORDER):
    activation, dale = key
    plot_selectivity_panel(ax, viz_data[key]["sel"])
    ax.set_title(f"{activation} — Dale={dale}")

axes[0, 0].set_ylabel("Units")
axes[1, 0].set_ylabel("Units")
fig.tight_layout()
fig.savefig(FIG_DIR / "selectivity_pcs.png", dpi=150, bbox_inches='tight')
plt.show()


# Shared colorbar / legend for the trajectory panel.
fig, ax = plt.subplots(figsize=(6, 0.5))
ax.axis("off")
primary_colors = np.array([[0.3, 0.4, 0.8], [0.8, 0.8, 0.8], [0.8, 0.2, 0.3]])
legend_cohs = np.linspace(-1, 1, 256)
gradient = np.array([
    interpolate_color(primary_colors[0], primary_colors[1], primary_colors[2], -1, 0, 1, v)
    for v in legend_cohs
])
ax.imshow(gradient[None, :, :], aspect="auto", extent=[-1, 1, 0, 1])
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
ax.set_yticks([])
ax.set_title("Coherence color scale (blue = -1, gray = 0, red = +1)")
fig.savefig(FIG_DIR / "coherence_color_scale.png", dpi=150, bbox_inches='tight')
plt.show()

from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(3, 0.5))
ax.axis("off")
ax.legend(
    handles=[
        Line2D([0], [0], marker="p", color="w", markerfacecolor="gray", markersize=10, label="motion context"),
        Line2D([0], [0], marker="*", color="w", markerfacecolor="gray", markersize=12, label="color context"),
    ],
    loc="center", ncol=2
)
ax.set_title("Marker shape")
fig.savefig(FIG_DIR / "marker_shape_legend.png", dpi=150, bbox_inches='tight')
plt.show()


## Compute distance matrices and MDS embeddings

Using the features extracted above, we build three $18 \times 18$ pairwise distance matrices (population trajectories, single-unit selectivities, trajectory endpoints) and embed each with 2-D MDS.

In [ ]:
D_traj = build_distance_matrix(traj_features, trajectory_distance)
D_sel = build_distance_matrix(sel_features, icp_distance)
D_end = build_distance_matrix(end_features, procrustes_distance)

mds_traj = MDS(n_components=2, dissimilarity="precomputed", random_state=0).fit_transform(D_traj)
mds_sel = MDS(n_components=2, dissimilarity="precomputed", random_state=0).fit_transform(D_sel)
mds_end = MDS(n_components=2, dissimilarity="precomputed", random_state=0).fit_transform(D_end)

print("Distance matrices computed: traj", D_traj.shape, "sel", D_sel.shape, "end", D_end.shape)


## Plot MDS embeddings

Each point is one trained network. Color = activation function; marker shape = Dale constraint. The expected qualitative pattern from the paper is that tanh networks cluster separately from relu/sigmoid networks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

colors = {"relu": "C0", "sigmoid": "C1", "tanh": "C2"}
markers = {False: "o", True: "^"}

for ax, emb, title in zip(
    axes.flat,
    [mds_traj, mds_sel, mds_end],
    ["Population trajectories", "Single-unit selectivity", "Trajectory endpoints"]
):
    for i, info in enumerate(models):
        ax.scatter(
            emb[i, 0], emb[i, 1],
            c=colors[info["activation"]],
            marker=markers[info["dale"]],
            s=100,
            edgecolors="k",
            linewidths=0.5,
            alpha=0.8,
        )
    ax.set_title(title)
    ax.set_xlabel("MDS 1")
    ax.set_ylabel("MDS 2")

# Shared legend below the panels.
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="C0", markersize=10, label="relu no-Dale"),
    Line2D([0], [0], marker="^", color="w", markerfacecolor="C0", markersize=10, label="relu Dale"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="C1", markersize=10, label="sigmoid no-Dale"),
    Line2D([0], [0], marker="^", color="w", markerfacecolor="C1", markersize=10, label="sigmoid Dale"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="C2", markersize=10, label="tanh no-Dale"),
    Line2D([0], [0], marker="^", color="w", markerfacecolor="C2", markersize=10, label="tanh Dale"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(FIG_DIR / "mds_embeddings.png", dpi=150, bbox_inches='tight')
plt.show()


## Summary

This notebook reproduced the CDDM experiments from Tolmachev & Engel (2025) inside the NeuralRNN framework. We trained 18 networks — six architectures (`{relu, sigmoid, tanh} × {Dale, no Dale}`) with three random seeds each — and analyzed their learned representations.

Delivered analyses:
- Dataset structure and example-trial visualization.
- Test-set decision accuracy (overall and split by context) plus masked MSE for every trained architecture.
- Population-trajectory and single-unit-selectivity visualizations as $2 \times 3$ panels, with one subplot per architecture and context/coherence separated by marker shape and color.
- Pairwise distance matrices and 2-D MDS embeddings for population trajectories, single-unit selectivities, and trajectory endpoints.

Fixed-point configuration and latent-circuit analyses are outside the scope of this notebook, but can be implemented by NeuralRNN (see notebook 01 and 02 for reference).